# ProMoNet speaker adaptation

Train or resume a speaker-adapted ProMoNet generator using WAV files from this project's `audio` folder. Completed models remain in ProMoNet's WSL run directory and automatically appear in the **Model** dropdown in `vowel_edit_pipeline.ipynb` after you click **Refresh models**.

Important:

- The default 100 adaptation steps are a fast workflow test, not a final-quality model.
- The step control is a **total adaptation target**. A 100-step run ends at generator step `800100`; set a target above 100 to continue farther.
- The current three files contain one sentence in three emotions and only about 5.7 seconds of audio. Use substantially more phonetic variety for a robust speaker model.
- TextGrids are not used for adaptation. ProMoNet performs its own training-feature preprocessing.

## 1. Imports and environment

Use the existing `Python (ProMoNet WSL)` kernel. This notebook does not start training until you press the button in the final section.

In [ ]:
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import os
import re
import traceback

import huggingface_hub
import ipywidgets as widgets
import pandas as pd
import soundfile
import torch
import torchaudio
import torchutil

import promonet

from IPython.display import clear_output, display

GPU = 0 if torch.cuda.is_available() else None
print("ProMoNet sample rate:", promonet.SAMPLE_RATE)
print("Pretrained step:", promonet.STEPS)
print("GPU argument:", GPU)
if GPU is not None:
    print("GPU:", torch.cuda.get_device_name(GPU))

## 2. Paths, audio inspection, and source manifests

In [ ]:
SHORT_DATA_WARNING_SECONDS = 60.0
SPEAKER_NAME_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_-]{0,63}$")


def find_project_root(start=None):
    '''Find the nearest parent containing the project audio directory.'''
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "audio").is_dir() and (candidate / "vowel_edit_pipeline").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find the ProMoNet project from {start}.")


def discover_adaptation_audio(audio_dir):
    '''Return original WAV files available for adaptation.'''
    files = sorted(
        path.resolve()
        for path in Path(audio_dir).glob("*.wav")
        if "-p" not in path.stem and "-l" not in path.stem
    )
    if not files:
        raise FileNotFoundError(f"No original WAV files found in {audio_dir}.")
    return files


def inspect_audio_files(files):
    '''Return basic audio metadata and reject silent or unreadable files.'''
    rows = []
    for file in files:
        waveform, sample_rate = torchaudio.load(file)
        if waveform.numel() == 0:
            raise ValueError(f"Audio is empty: {file}")
        peak = waveform.abs().max().item()
        if peak == 0:
            raise ValueError(f"Audio is silent: {file}")
        rows.append({
            "file": Path(file).name,
            "path": str(Path(file).resolve()),
            "channels": int(waveform.shape[0]),
            "sample_rate": int(sample_rate),
            "duration_s": waveform.shape[-1] / sample_rate,
            "peak": peak,
        })
    return pd.DataFrame(rows)


def validate_speaker_name(name):
    '''Validate a filesystem-safe adaptation run name.'''
    name = str(name).strip()
    if not SPEAKER_NAME_PATTERN.fullmatch(name):
        raise ValueError(
            "Speaker name must start with a letter or digit, contain only "
            "letters, digits, underscores, or hyphens, and be at most 64 characters."
        )
    if name.casefold() in {"pretrained", "custom", "none"}:
        raise ValueError(f"Reserved speaker name: {name}")
    return name


def file_sha256(path, chunk_size=1024 * 1024):
    '''Hash one source recording without loading it all into memory.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def build_source_manifest(files):
    '''Build deterministic source records used to authorize resume.'''
    records = []
    for path in sorted((Path(file).resolve() for file in files), key=str):
        stat = path.stat()
        records.append({
            "path": str(path),
            "size_bytes": int(stat.st_size),
            "modified_ns": int(stat.st_mtime_ns),
            "sha256": file_sha256(path),
        })
    return records


def source_manifests_match(first, second):
    '''Compare source identity while ignoring modification-time metadata.'''
    identity_fields = ("path", "size_bytes", "sha256")
    normalize = lambda rows: [
        {field: row[field] for field in identity_fields}
        for row in rows
    ]
    return normalize(first) == normalize(second)


PROJECT_ROOT = find_project_root()
AUDIO_DIR = PROJECT_ROOT / "audio"
PIPELINE_DIR = PROJECT_ROOT / "vowel_edit_pipeline"
MANIFEST_DIR = PIPELINE_DIR / "adaptation_manifests"
ADAPTATION_FILES = discover_adaptation_audio(AUDIO_DIR)

AUDIO_TABLE = inspect_audio_files(ADAPTATION_FILES)
display(AUDIO_TABLE[["file", "channels", "sample_rate", "duration_s", "peak"]])
TOTAL_DURATION_SECONDS = AUDIO_TABLE["duration_s"].sum()
print(f"Total available adaptation audio: {TOTAL_DURATION_SECONDS:.2f} seconds")
if TOTAL_DURATION_SECONDS < SHORT_DATA_WARNING_SECONDS:
    print(
        "WARNING: This is a very small adaptation set. It is appropriate for a "
        "workflow test, but not for a robust speaker model."
    )

## 3. Checkpoint and manifest management

In [ ]:
def checkpoint_step(path):
    '''Parse the numerical training step from a checkpoint filename.'''
    match = re.search(r"-(\d+)\.pt$", Path(path).name)
    if match is None:
        raise ValueError(f"Unrecognized checkpoint filename: {path}")
    return int(match.group(1))


def latest_checkpoint_pair(speaker_name):
    '''Return the latest generator/discriminator pair sharing the same step.'''
    speaker_name = validate_speaker_name(speaker_name)
    run_directory = promonet.RUNS_DIR / promonet.CONFIG / "adapt" / speaker_name
    generators = {
        checkpoint_step(path): path
        for path in run_directory.glob("generator-*.pt")
    }
    discriminators = {
        checkpoint_step(path): path
        for path in run_directory.glob("discriminator-*.pt")
    }
    common_steps = sorted(set(generators) & set(discriminators))
    if not common_steps:
        return {
            "speaker_name": speaker_name,
            "run_directory": run_directory,
            "step": None,
            "adaptation_steps": 0,
            "generator": None,
            "discriminator": None,
        }
    step = common_steps[-1]
    return {
        "speaker_name": speaker_name,
        "run_directory": run_directory,
        "step": step,
        "adaptation_steps": max(0, step - int(promonet.STEPS)),
        "generator": generators[step],
        "discriminator": discriminators[step],
    }


def discover_adapted_models():
    '''Discover the numerically latest generator for every adapted speaker.'''
    root = promonet.RUNS_DIR / promonet.CONFIG / "adapt"
    models = []
    if not root.is_dir():
        return models
    for speaker_directory in sorted(path for path in root.iterdir() if path.is_dir()):
        generators = list(speaker_directory.glob("generator-*.pt"))
        if not generators:
            continue
        latest = max(generators, key=checkpoint_step)
        manifest_exists = (MANIFEST_DIR / f"{speaker_directory.name}.json").is_file()
        models.append({
            "speaker_name": speaker_directory.name,
            "generator": latest,
            "step": checkpoint_step(latest),
            "adaptation_steps": max(0, checkpoint_step(latest) - int(promonet.STEPS)),
            "manifest_status": "managed" if manifest_exists else "legacy",
        })
    return models


def manifest_path(speaker_name):
    return MANIFEST_DIR / f"{validate_speaker_name(speaker_name)}.json"


def load_run_manifest(speaker_name):
    path = manifest_path(speaker_name)
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def save_run_manifest(manifest):
    '''Atomically update one adaptation manifest.'''
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    destination = manifest_path(manifest["speaker_name"])
    temporary = destination.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    temporary.replace(destination)
    return destination


def check_resume_permission(speaker_name, source_files):
    '''Allow new runs or matching managed runs; reject legacy and changed data.'''
    source_records = build_source_manifest(source_files)
    checkpoint_pair = latest_checkpoint_pair(speaker_name)
    manifest = load_run_manifest(speaker_name)
    if checkpoint_pair["generator"] is not None and manifest is None:
        raise ValueError(
            f"{speaker_name!r} is a legacy run without a source manifest. "
            "It is available for synthesis, but automatic resume is unsafe. "
            "Choose a new speaker name."
        )
    if manifest is not None and not source_manifests_match(
        manifest["source_files"], source_records
    ):
        raise ValueError(
            f"The selected files differ from the saved manifest for {speaker_name!r}. "
            "Choose a new speaker name instead of mixing datasets."
        )
    return source_records, checkpoint_pair, manifest


def adaptation_artifact_roots():
    '''Return the only roots from which adaptation cleanup is permitted.'''
    return {
        "run": promonet.RUNS_DIR / promonet.CONFIG / "adapt",
        "cache": promonet.CACHE_DIR,
        "manifest": MANIFEST_DIR,
        "partition": promonet.ASSETS_DIR / "partitions" / "adaptation",
    }


ADAPTATION_ARTIFACT_ROOTS = adaptation_artifact_roots()


def adaptation_artifact_targets(speaker_name):
    '''Return the four exact artifacts owned by one adaptation run.'''
    speaker_name = validate_speaker_name(speaker_name)
    roots = ADAPTATION_ARTIFACT_ROOTS
    return {
        "run": roots["run"] / speaker_name,
        "cache": roots["cache"] / speaker_name,
        "manifest": roots["manifest"] / f"{speaker_name}.json",
        "partition": roots["partition"] / f"{speaker_name}.json",
    }


def _validated_cleanup_target(path, root):
    '''Lexically contain one exact target without following its final symlink.'''
    root = Path(root).resolve()
    target = Path(os.path.abspath(path))
    try:
        relative = target.relative_to(root)
    except ValueError as error:
        raise ValueError(f"Cleanup target escapes its permitted root: {target}") from error
    if not relative.parts:
        raise ValueError(f"Refusing to remove an artifact root: {root}")
    return target


def _path_inventory(path):
    '''List files and symlinks under a target without following symlinks.'''
    path = Path(path)
    if not path.exists() and not path.is_symlink():
        return []
    if path.is_symlink() or not path.is_dir():
        return [{"path": str(path), "size_bytes": int(path.lstat().st_size)}]

    entries = []
    for current, directories, filenames in os.walk(path, followlinks=False):
        current = Path(current)
        for name in sorted(filenames):
            child = current / name
            entries.append({"path": str(child), "size_bytes": int(child.lstat().st_size)})
        for name in sorted(directories):
            child = current / name
            if child.is_symlink():
                entries.append({"path": str(child), "size_bytes": int(child.lstat().st_size)})
    return entries


def _looks_like_adaptation_cache(path):
    '''Recognize the numbered WAV staging layout created by this notebook.'''
    path = Path(path)
    if not path.is_dir():
        return False
    return any(
        child.is_file() and re.fullmatch(r"\d{6}-100\.wav", child.name)
        for child in path.iterdir()
    )


def _looks_like_adaptation_partition(path):
    '''Recognize only speaker partitions created by this notebook.'''
    path = Path(path)
    if not path.is_file():
        return False
    try:
        partition = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        isinstance(partition, dict)
        and isinstance(partition.get("train-adapt"), list)
        and isinstance(partition.get("valid-adapt"), list)
    )


def discover_adaptation_artifacts():
    '''Discover complete, legacy, failed, and partial adaptation artifacts.'''
    roots = ADAPTATION_ARTIFACT_ROOTS
    names = set()
    if roots["run"].is_dir():
        names.update(path.name for path in roots["run"].iterdir() if path.is_dir())
    if roots["manifest"].is_dir():
        names.update(path.stem for path in roots["manifest"].glob("*.json"))
    if roots["partition"].is_dir():
        names.update(
            path.stem for path in roots["partition"].glob("*.json")
            if _looks_like_adaptation_partition(path)
        )
    if roots["cache"].is_dir():
        names.update(
            path.name for path in roots["cache"].iterdir()
            if _looks_like_adaptation_cache(path)
        )

    artifacts = []
    for name in sorted(names):
        try:
            targets = adaptation_artifact_targets(name)
        except ValueError:
            continue
        existing = {
            kind: path for kind, path in targets.items()
            if path.exists() or path.is_symlink()
        }
        generators = []
        if targets["run"].is_dir():
            for generator in targets["run"].glob("generator-*.pt"):
                try:
                    generators.append((checkpoint_step(generator), generator))
                except ValueError:
                    pass
        latest_step = max((step for step, _ in generators), default=None)

        manifest = None
        if targets["manifest"].is_file():
            try:
                manifest = json.loads(targets["manifest"].read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                manifest = {"status": "invalid_manifest"}
        if manifest is not None:
            status = str(manifest.get("status", "managed"))
        elif latest_step is not None:
            status = "legacy"
        elif "run" in existing:
            status = "partial"
        else:
            status = "orphaned"

        inventories = [_path_inventory(path) for path in existing.values()]
        files = [entry for inventory in inventories for entry in inventory]
        artifacts.append({
            "speaker_name": name,
            "status": status,
            "latest_step": latest_step,
            "artifact_types": ", ".join(existing),
            "file_count": len(files),
            "total_bytes": sum(entry["size_bytes"] for entry in files),
        })
    return artifacts


def preview_adaptation_reset(speaker_name):
    '''Return exact targets and file inventory for one permanent reset.'''
    speaker_name = validate_speaker_name(speaker_name)
    targets = adaptation_artifact_targets(speaker_name)
    target_records = []
    all_files = []
    for kind, path in targets.items():
        path = _validated_cleanup_target(path, ADAPTATION_ARTIFACT_ROOTS[kind])
        if not path.exists() and not path.is_symlink():
            continue
        inventory = _path_inventory(path)
        target_records.append({
            "artifact_type": kind,
            "path": str(path),
            "file_count": len(inventory),
            "total_bytes": sum(entry["size_bytes"] for entry in inventory),
        })
        all_files.extend(
            {"artifact_type": kind, **entry} for entry in inventory
        )
    return {
        "speaker_name": speaker_name,
        "targets": target_records,
        "files": all_files,
        "file_count": len(all_files),
        "total_bytes": sum(entry["size_bytes"] for entry in all_files),
    }


def _permanently_remove_target(path):
    '''Remove one validated target without following directory symlinks.'''
    path = Path(path)
    if path.is_symlink() or not path.is_dir():
        path.unlink()
        return
    for current, directories, filenames in os.walk(path, topdown=False, followlinks=False):
        current = Path(current)
        for name in filenames:
            (current / name).unlink()
        for name in directories:
            child = current / name
            if child.is_symlink():
                child.unlink()
            else:
                child.rmdir()
    path.rmdir()


def reset_adaptation_run(speaker_name, confirmation):
    '''Permanently remove one run after exact-name confirmation.'''
    speaker_name = validate_speaker_name(speaker_name)
    if confirmation != speaker_name:
        raise ValueError("Confirmation must exactly match the selected speaker name.")
    preview = preview_adaptation_reset(speaker_name)
    if not preview["targets"]:
        raise FileNotFoundError(f"No adaptation artifacts exist for {speaker_name!r}.")

    removed = []
    for record in preview["targets"]:
        kind = record["artifact_type"]
        path = _validated_cleanup_target(record["path"], ADAPTATION_ARTIFACT_ROOTS[kind])
        _permanently_remove_target(path)
        removed.append(str(path))
    return {
        "speaker_name": speaker_name,
        "removed_targets": removed,
        "removed_files": preview["file_count"],
        "reclaimed_bytes": preview["total_bytes"],
    }


def format_bytes(number):
    value = float(number)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.1f} {unit}"
        value /= 1024


EXISTING_MODELS = pd.DataFrame(discover_adapted_models())
if EXISTING_MODELS.empty:
    print("No adapted generators found yet.")
else:
    display(EXISTING_MODELS)

## 4. Compatibility layer for the installed ProMoNet

In [ ]:
COMPATIBILITY_REVISION = "2026-08-22-managed-adaptation-v1"


def install_torchaudio_info_compatibility():
    '''Restore metadata access expected by ProMoNet 0.0.1 dependencies.'''
    if hasattr(torchaudio, "info"):
        return

    def torchaudio_info_compatibility(file, *args, **kwargs):
        metadata = soundfile.info(str(file))
        bits_per_sample = {
            "PCM_U8": 8, "PCM_16": 16, "PCM_24": 24,
            "PCM_32": 32, "FLOAT": 32, "DOUBLE": 64,
        }.get(metadata.subtype, 0)
        return SimpleNamespace(
            sample_rate=int(metadata.samplerate),
            num_frames=int(metadata.frames),
            num_channels=int(metadata.channels),
            bits_per_sample=bits_per_sample,
            encoding=str(metadata.subtype),
        )

    torchaudio.info = torchaudio_info_compatibility


def prepare_global_features_with_neutral_ratios(
    self,
    speakers,
    spectral_balance_ratios,
    loudness_ratios,
):
    '''Preserve the pretrained generator's global conditioning layout.'''
    global_features = self.speaker_embedding(speakers).unsqueeze(-1)
    expected_ratio_channels = promonet.GLOBAL_CHANNELS - promonet.SPEAKER_CHANNELS
    if expected_ratio_channels == 0:
        return global_features
    if expected_ratio_channels != 2:
        raise RuntimeError(
            "Expected zero or two global ratio channels, found "
            f"{expected_ratio_channels}."
        )
    if not promonet.AUGMENT_PITCH:
        spectral_balance_ratios = torch.ones_like(spectral_balance_ratios)
    if not promonet.AUGMENT_LOUDNESS:
        loudness_ratios = torch.ones_like(loudness_ratios)
    spectral_balance_ratios = spectral_balance_ratios.to(
        device=global_features.device, dtype=global_features.dtype
    )
    loudness_ratios = loudness_ratios.to(
        device=global_features.device, dtype=global_features.dtype
    )
    result = torch.cat((
        global_features,
        spectral_balance_ratios[:, None, None],
        loudness_ratios[:, None, None],
    ), dim=1)
    if result.shape[1] != promonet.GLOBAL_CHANNELS:
        raise RuntimeError(
            f"Prepared {result.shape[1]} channels; expected {promonet.GLOBAL_CHANNELS}."
        )
    return result


def install_promonet_adaptation_compatibility():
    '''Install the tested ProMoNet 0.0.1 adaptation wrapper in this kernel.'''
    install_torchaudio_info_compatibility()
    prepare_global_features_with_neutral_ratios._promonet_neutral_ratios = True
    promonet.model.Generator.prepare_global_features = (
        prepare_global_features_with_neutral_ratios
    )
    if getattr(promonet.adapt.speaker, "_compatibility_revision", None) == COMPATIBILITY_REVISION:
        return

    def compatible_speaker(name, files, checkpoint=None, gpu=None):
        files = [Path(file) for file in files]
        if not files:
            raise ValueError("Speaker adaptation requires at least one audio file.")
        if promonet.AUGMENT_PITCH or promonet.AUGMENT_LOUDNESS:
            raise ValueError("Disable pitch and loudness augmentation for this workflow.")

        cache = promonet.CACHE_DIR / name
        cache.mkdir(exist_ok=True, parents=True)
        staged_files = []
        for index, audio_file in enumerate(files):
            audio = promonet.load.audio(audio_file)
            maximum = torch.abs(audio).max()
            if maximum == 0:
                raise ValueError(f"Adaptation audio is silent: {audio_file}")
            if maximum < 0.35:
                audio = audio * (0.35 / maximum)
            staged_file = cache / f"{index:06d}-100.wav"
            torchaudio.save(staged_file, audio, promonet.SAMPLE_RATE)
            staged_files.append(staged_file)

        promonet.data.preprocess.datasets([name], gpu=gpu)
        for staged_file in staged_files:
            processed_text = staged_file.with_suffix(".txt")
            base_name = staged_file.stem.removesuffix("-100")
            (cache / f"{base_name}.txt").write_text(
                processed_text.read_text(encoding="utf-8"), encoding="utf-8"
            )

        stems = [file.stem.removesuffix("-100") for file in staged_files]
        partition_directory = promonet.ASSETS_DIR / "partitions" / "adaptation"
        partition_directory.mkdir(parents=True, exist_ok=True)
        partition_file = partition_directory / f"{name}.json"
        partition_file.write_text(
            json.dumps({"train-adapt": stems, "valid-adapt": stems[:1]}, indent=4),
            encoding="utf-8",
        )

        directory = promonet.RUNS_DIR / promonet.CONFIG / "adapt" / name
        directory.mkdir(exist_ok=True, parents=True)
        generator_path = torchutil.checkpoint.latest_path(directory, "generator-*.pt")
        discriminator_path = torchutil.checkpoint.latest_path(
            directory, "discriminator-*.pt"
        )
        if generator_path and discriminator_path:
            checkpoint = directory
        if checkpoint is None:
            generator_checkpoint = huggingface_hub.hf_hub_download(
                "maxrmorrison/promonet",
                f"generator-00{promonet.STEPS}.pt",
            )
            huggingface_hub.hf_hub_download(
                "maxrmorrison/promonet",
                f"discriminator-00{promonet.STEPS}.pt",
            )
            checkpoint = Path(generator_checkpoint).parent

        promonet.train(
            directory,
            name,
            train_partition="train-adapt",
            valid_partition="valid-adapt",
            adapt_from=checkpoint,
            gpu=gpu,
        )
        return torchutil.checkpoint.latest_path(directory, "generator-*.pt")

    compatible_speaker._compatibility_revision = COMPATIBILITY_REVISION
    promonet.adapt.speaker = compatible_speaker


def adaptation_compatibility_preflight():
    '''Verify the compatibility patches without preprocessing or training.'''
    install_promonet_adaptation_compatibility()
    original_pitch = promonet.AUGMENT_PITCH
    original_loudness = promonet.AUGMENT_LOUDNESS
    try:
        promonet.AUGMENT_PITCH = False
        promonet.AUGMENT_LOUDNESS = False
        probe = SimpleNamespace(
            speaker_embedding=torch.nn.Embedding(1, promonet.SPEAKER_CHANNELS)
        )
        features = promonet.model.Generator.prepare_global_features(
            probe,
            torch.zeros(1, dtype=torch.long),
            torch.ones(1),
            torch.ones(1),
        )
        expected = (1, promonet.GLOBAL_CHANNELS, 1)
        if features.shape != expected:
            raise RuntimeError(f"Conditioning shape {features.shape}; expected {expected}.")
        return tuple(features.shape)
    finally:
        promonet.AUGMENT_PITCH = original_pitch
        promonet.AUGMENT_LOUDNESS = original_loudness


CONDITIONING_SHAPE = adaptation_compatibility_preflight()
print("Compatibility revision:", COMPATIBILITY_REVISION)
print("torchaudio.info available:", hasattr(torchaudio, "info"))
print("Conditioning preflight shape:", CONDITIONING_SHAPE)

## 5. Managed training

In [ ]:
def run_managed_adaptation(
    speaker_name,
    files,
    target_adaptation_steps=100,
    checkpoint_interval=100,
    gpu=GPU,
):
    '''Train, resume, or return an already-complete managed adaptation run.'''
    speaker_name = validate_speaker_name(speaker_name)
    files = [Path(file).resolve() for file in files]
    if not files:
        raise ValueError("Select at least one adaptation WAV.")
    if int(target_adaptation_steps) <= 0:
        raise ValueError("Target adaptation steps must be positive.")
    if int(checkpoint_interval) <= 0:
        raise ValueError("Checkpoint interval must be positive.")

    source_records, current, existing_manifest = check_resume_permission(
        speaker_name, files
    )
    target_adaptation_steps = int(target_adaptation_steps)
    if current["generator"] is not None and target_adaptation_steps <= current["adaptation_steps"]:
        print(
            f"Run already reached {current['adaptation_steps']} adaptation steps; "
            f"target is {target_adaptation_steps}. No training is needed."
        )
        return current

    manifest = existing_manifest or {
        "schema_version": 1,
        "speaker_name": speaker_name,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_files": source_records,
    }
    manifest.update({
        "status": "running",
        "updated_at": datetime.now().isoformat(timespec="seconds"),
        "target_adaptation_steps": target_adaptation_steps,
        "checkpoint_interval": int(checkpoint_interval),
        "run_directory": str(current["run_directory"]),
        "completed_adaptation_steps": int(current["adaptation_steps"]),
        "generator": None if current["generator"] is None else str(current["generator"]),
        "discriminator": None if current["discriminator"] is None else str(current["discriminator"]),
    })
    save_run_manifest(manifest)

    setting_names = (
        "AUGMENT_PITCH", "AUGMENT_LOUDNESS", "CHECKPOINT_INTERVAL",
        "ADAPTATION_STEPS", "BATCH_SIZE", "NUM_WORKERS",
    )
    original_settings = {name: getattr(promonet, name) for name in setting_names}
    install_promonet_adaptation_compatibility()
    try:
        promonet.AUGMENT_PITCH = False
        promonet.AUGMENT_LOUDNESS = False
        promonet.CHECKPOINT_INTERVAL = int(checkpoint_interval)
        promonet.ADAPTATION_STEPS = target_adaptation_steps
        promonet.BATCH_SIZE = 1
        promonet.NUM_WORKERS = 1
        print("Speaker:", speaker_name)
        print("Files:", len(files))
        print("Target adaptation steps:", target_adaptation_steps)
        print("Resume from adaptation step:", current["adaptation_steps"])
        promonet.adapt.speaker(
            name=speaker_name,
            files=files,
            checkpoint=None,
            gpu=gpu,
        )
        completed = latest_checkpoint_pair(speaker_name)
        manifest.update({
            "status": "complete",
            "updated_at": datetime.now().isoformat(timespec="seconds"),
            "completed_adaptation_steps": completed["adaptation_steps"],
            "generator": str(completed["generator"]),
            "discriminator": str(completed["discriminator"]),
        })
        save_run_manifest(manifest)
        return completed
    except Exception as error:
        manifest.update({
            "status": "failed",
            "updated_at": datetime.now().isoformat(timespec="seconds"),
            "error": f"{type(error).__name__}: {error}",
        })
        save_run_manifest(manifest)
        raise
    finally:
        for name, value in original_settings.items():
            setattr(promonet, name, value)

## 6. Lightweight self-checks

These checks do not preprocess audio or start training. They validate any nonempty adaptation dataset and inspect all existing checkpoints read-only, without assuming a fixed file count, duration, speaker name, or checkpoint step.

In [ ]:
def run_adaptation_self_checks():
    # Validate dataset invariants instead of assuming a fixed number of files.
    assert ADAPTATION_FILES, "No adaptation WAV files were discovered."
    assert len(AUDIO_TABLE) == len(ADAPTATION_FILES)
    assert float(TOTAL_DURATION_SECONDS) > 0.0
    assert abs(
        float(TOTAL_DURATION_SECONDS) - float(AUDIO_TABLE["duration_s"].sum())
    ) < 1e-6
    assert (AUDIO_TABLE["duration_s"] > 0).all()
    assert (AUDIO_TABLE["sample_rate"] > 0).all()
    assert (AUDIO_TABLE["channels"] > 0).all()
    assert (AUDIO_TABLE["peak"] > 0).all()

    discovered_paths = {str(Path(path).resolve()) for path in ADAPTATION_FILES}
    assert set(AUDIO_TABLE["path"]) == discovered_paths

    assert validate_speaker_name("speaker_01") == "speaker_01"
    for invalid_name in ("", "bad name", "../speaker", "_speaker"):
        try:
            validate_speaker_name(invalid_name)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid speaker name was accepted: {invalid_name!r}")

    source_records = build_source_manifest(ADAPTATION_FILES)
    assert len(source_records) == len(ADAPTATION_FILES)
    assert [record["path"] for record in source_records] == sorted(discovered_paths)
    assert all(len(record["sha256"]) == 64 for record in source_records)
    assert source_manifests_match(source_records, source_records)
    changed = json.loads(json.dumps(source_records))
    changed[0]["sha256"] = "0" * 64
    assert not source_manifests_match(source_records, changed)
    assert checkpoint_step(Path("generator-00800100.pt")) == 800100
    assert CONDITIONING_SHAPE == (1, promonet.GLOBAL_CHANNELS, 1)

    models = discover_adapted_models()
    for model in models:
        generator = Path(model["generator"])
        assert generator.is_file()
        assert checkpoint_step(generator) == int(model["step"])
        assert int(model["adaptation_steps"]) == max(
            0, int(model["step"]) - int(promonet.STEPS)
        )
        assert model["manifest_status"] in {"managed", "legacy"}
    return {
        "audio_files": len(ADAPTATION_FILES),
        "total_duration_s": round(float(TOTAL_DURATION_SECONDS), 3),
        "short_data_warning": bool(
            TOTAL_DURATION_SECONDS < SHORT_DATA_WARNING_SECONDS
        ),
        "conditioning_shape": CONDITIONING_SHAPE,
        "existing_models": len(models),
        "model_names": [model["speaker_name"] for model in models],
        "latest_model_steps": {
            model["speaker_name"]: int(model["step"]) for model in models
        },
    }


SELF_CHECK_RESULTS = run_adaptation_self_checks()
display(SELF_CHECK_RESULTS)

## 7. Training controls

In [ ]:
def build_adaptation_controls():
    '''Build audio-selection, target-step, and train/resume controls.'''
    speaker_input = widgets.Text(
        value="applab_47_10_v1",
        description="Speaker name:",
        layout=widgets.Layout(width="600px"),
    )
    file_select = widgets.SelectMultiple(
        options=[(path.name, str(path)) for path in ADAPTATION_FILES],
        value=tuple(str(path) for path in ADAPTATION_FILES),
        description="Audio:",
        rows=max(5, len(ADAPTATION_FILES)),
        layout=widgets.Layout(width="700px"),
    )
    target_steps = widgets.BoundedIntText(
        value=100,
        min=1,
        max=100000,
        description="Target steps:",
    )
    checkpoint_interval = widgets.BoundedIntText(
        value=100,
        min=1,
        max=100000,
        description="Checkpoint every:",
    )
    train_button = widgets.Button(
        description="Train or resume adaptation",
        button_style="warning",
        icon="play",
    )
    refresh_button = widgets.Button(description="Refresh model list", icon="refresh")
    cleanup_dropdown = widgets.Dropdown(
        description="Run to reset:",
        layout=widgets.Layout(width="800px"),
    )
    cleanup_refresh_button = widgets.Button(
        description="Refresh cleanup list", icon="refresh"
    )
    cleanup_confirmation = widgets.Text(
        description="Type run name:",
        placeholder="Exact selected speaker name",
        layout=widgets.Layout(width="800px"),
    )
    cleanup_button = widgets.Button(
        description="Permanently reset selected run",
        button_style="danger",
        icon="trash",
        disabled=True,
    )
    model_output = widgets.Output()
    training_output = widgets.Output()
    cleanup_preview_output = widgets.Output(
        layout=widgets.Layout(
            border="1px solid #bbb", max_height="300px", overflow_y="auto"
        )
    )
    cleanup_result_output = widgets.Output()
    state = {"last_result": None, "busy": False, "artifacts": [], "preview": None}

    def selected_cleanup_name():
        return cleanup_dropdown.value

    def update_cleanup_button(*_):
        selected = selected_cleanup_name()
        cleanup_button.disabled = bool(
            state["busy"]
            or selected is None
            or cleanup_confirmation.value != selected
        )

    def set_busy(busy):
        state["busy"] = bool(busy)
        for control in (
            speaker_input, file_select, target_steps, checkpoint_interval,
            train_button, refresh_button, cleanup_dropdown,
            cleanup_refresh_button, cleanup_confirmation,
        ):
            control.disabled = state["busy"]
        update_cleanup_button()

    def refresh_cleanup_preview(*_):
        selected = selected_cleanup_name()
        cleanup_confirmation.value = ""
        state["preview"] = None
        with cleanup_preview_output:
            clear_output(wait=True)
            if selected is None:
                print("No adaptation artifacts found.")
                update_cleanup_button()
                return
            try:
                preview = preview_adaptation_reset(selected)
                state["preview"] = preview
                print(f"Permanent reset preview for {selected!r}")
                print(f"Files: {preview['file_count']}")
                print(f"Disk usage: {format_bytes(preview['total_bytes'])}")
                print("Exact artifact targets:")
                for target in preview["targets"]:
                    print(
                        f" - {target['artifact_type']}: {target['path']} "
                        f"({target['file_count']} files, {format_bytes(target['total_bytes'])})"
                    )
                print("Exact files and symlinks:")
                for entry in preview["files"]:
                    print(f" - [{entry['artifact_type']}] {entry['path']}")
            except Exception as error:
                print(f"Could not preview cleanup: {error}")
        update_cleanup_button()

    def refresh_cleanup_options(_=None):
        previous = selected_cleanup_name()
        artifacts = discover_adaptation_artifacts()
        state["artifacts"] = artifacts
        cleanup_dropdown.options = [
            (
                f"{item['speaker_name']} — {item['status']} — "
                f"{format_bytes(item['total_bytes'])} — {item['artifact_types']}",
                item["speaker_name"],
            )
            for item in artifacts
        ]
        available = {item["speaker_name"] for item in artifacts}
        if previous in available:
            cleanup_dropdown.value = previous
        refresh_cleanup_preview()

    def refresh_models(_=None):
        with model_output:
            clear_output(wait=True)
            models = pd.DataFrame(discover_adapted_models())
            if models.empty:
                print("No adapted generators found.")
            else:
                display(models)

    def train_clicked(_):
        set_busy(True)
        with training_output:
            clear_output(wait=True)
            try:
                selected_files = [Path(path) for path in file_select.value]
                selected_table = inspect_audio_files(selected_files)
                display(selected_table[["file", "sample_rate", "duration_s", "peak"]])
                selected_duration = selected_table["duration_s"].sum()
                print(f"Selected duration: {selected_duration:.2f} seconds")
                if selected_duration < SHORT_DATA_WARNING_SECONDS:
                    print("WARNING: The selected adaptation data is very small.")
                result = run_managed_adaptation(
                    speaker_name=speaker_input.value,
                    files=selected_files,
                    target_adaptation_steps=target_steps.value,
                    checkpoint_interval=checkpoint_interval.value,
                )
                state["last_result"] = result
                print("Adaptation result:")
                print(" Generator:", result["generator"])
                print(" Discriminator:", result["discriminator"])
                print(" Completed adaptation steps:", result["adaptation_steps"])
                print("Open the vowel editor and click Refresh models.")
                refresh_models()
            except Exception as error:
                print(f"Adaptation failed: {error}")
                traceback.print_exc(limit=2)
            finally:
                set_busy(False)
                refresh_cleanup_options()

    def cleanup_clicked(_):
        selected = selected_cleanup_name()
        confirmation = cleanup_confirmation.value
        set_busy(True)
        with cleanup_result_output:
            clear_output(wait=True)
            try:
                result = reset_adaptation_run(selected, confirmation)
                last_result = state.get("last_result")
                if last_result and last_result.get("speaker_name") == selected:
                    state["last_result"] = None
                print(f"Permanently reset {selected!r}.")
                print(f"Removed files: {result['removed_files']}")
                print(f"Reclaimed: {format_bytes(result['reclaimed_bytes'])}")
                for path in result["removed_targets"]:
                    print(f" - removed: {path}")
                print("Open the vowel editor and click Refresh models.")
                refresh_models()
            except Exception as error:
                print(f"Cleanup failed: {error}")
                traceback.print_exc(limit=2)
            finally:
                refresh_cleanup_options()
                set_busy(False)

    train_button.on_click(train_clicked)
    refresh_button.on_click(refresh_models)
    cleanup_refresh_button.on_click(refresh_cleanup_options)
    cleanup_dropdown.observe(refresh_cleanup_preview, names="value")
    cleanup_confirmation.observe(update_cleanup_button, names="value")
    cleanup_button.on_click(cleanup_clicked)
    display(widgets.VBox([
        widgets.HTML("<h3>Adaptation data and run identity</h3>"),
        speaker_input,
        file_select,
        widgets.HTML("<h3>Training target</h3>"),
        target_steps,
        checkpoint_interval,
        widgets.HTML("Batch size: <b>1</b> &nbsp; Workers: <b>1</b> &nbsp; Augmentation: <b>disabled</b>"),
        train_button,
        training_output,
        widgets.HTML("<h3>Available adapted models</h3>"),
        refresh_button,
        model_output,
        widgets.HTML("<h3>Manage adaptation runs</h3>"),
        widgets.HTML(
            "<b style='color:#b00020'>Permanent deletion:</b> Shut down any other "
            "kernel training this speaker. A reset removes its checkpoints, cache, "
            "manifest, and partition and cannot be undone."
        ),
        widgets.HBox([cleanup_dropdown, cleanup_refresh_button]),
        cleanup_preview_output,
        cleanup_confirmation,
        cleanup_button,
        cleanup_result_output,
    ]))
    refresh_models()
    refresh_cleanup_options()
    return {
        "speaker_name": speaker_input,
        "files": file_select,
        "target_steps": target_steps,
        "checkpoint_interval": checkpoint_interval,
        "train_button": train_button,
        "refresh_button": refresh_button,
        "cleanup_dropdown": cleanup_dropdown,
        "cleanup_refresh_button": cleanup_refresh_button,
        "cleanup_confirmation": cleanup_confirmation,
        "cleanup_button": cleanup_button,
        "cleanup_preview": cleanup_preview_output,
        "cleanup_result": cleanup_result_output,
        "state": state,
    }


CONTROLS = build_adaptation_controls()

## Using the trained model

1. Finish training and note the reported generator path.
2. Open `vowel_edit_pipeline.ipynb` with the same WSL environment.
3. Click **Refresh models** in its synthesis controls.
4. Choose `Adapted: <speaker name> — generator-<step>` from the Model dropdown. The editor automatically sets speaker ID `0`.

Existing runs without manifests, including `applab_47_16`, are safe to use for synthesis but intentionally cannot be resumed through this notebook. Choose a new name to begin a managed run. The **Manage adaptation runs** section can permanently reset one complete, legacy, failed, or partial speaker run after you preview every artifact and type the exact speaker name. Shut down other training kernels first; deletion cannot be undone.